# Autonomous Data Analyst Agent (Fabric + ReAct + RAG)

# Step 1: Install Dependencies

In [ ]:
!pip install openai==0.28.1 langchain==0.1.16 langchain-community wikipedia requests

# Step 2: Setup OpenAI

In [ ]:
import openai
openai.api_key = "YOUR_API_KEY"

# Step 3: LLM Function

In [ ]:
def ask_llm(prompt):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful data analyst."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response["choices"][0]["message"]["content"]

# Step 4: Generate SQL

In [ ]:
def generate_sql(question):
    prompt = f"""
    You are a data analyst working with Fabric.
    Table: gold.Fact_tickets

    Rules:
    - Use CreatedAt for date
    - Use DATE(CreatedAt) for filtering
    - Return only SQL query

    Question: {question}
    """
    return ask_llm(prompt)

# Step 5: Clean SQL

In [ ]:
def clean_sql(sql):
    return sql.replace("```sql", "").replace("```", "").strip()

# Step 6: Query Fabric

In [ ]:
def query_fabric(sql):
    return spark.sql(sql).toPandas().to_string()

# Step 7: Python Analysis Tool

In [ ]:
def python_analysis_tool(data):
    prompt = f"""
    Analyze the data and provide trends:
    {data}
    """
    return ask_llm(prompt)

# Step 8: RAG Tool

In [ ]:
def rag_tool(query):
    return "Internal insights based on documents"

# Step 9: Wikipedia Tool

In [ ]:
import requests

def wiki_tool(query):
    query = query.replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query}"
    res = requests.get(url)
    data = res.json()
    return data.get("extract", "No info found")

# Step 10: LangChain Tools

In [ ]:
from langchain.agents import Tool

sql_tool = Tool(name="SQL", func=lambda q: query_fabric(clean_sql(generate_sql(q))), description="Fetch ticket data")

python_tool = Tool(name="Python", func=lambda data: python_analysis_tool(data), description="Analyze data")

rag_tool_lc = Tool(name="RAG", func=lambda q: rag_tool(q), description="Internal knowledge")

wiki_tool_lc = Tool(name="Wikipedia", func=lambda q: wiki_tool(q), description="External explanations")

# Step 11: Initialize Agent

In [ ]:
from langchain.agents import initialize_agent

llm = CustomOpenAI()

tools = [sql_tool, python_tool, rag_tool_lc, wiki_tool_lc]

agent = initialize_agent(
    tools,
    llm,
    agent="zero-shot-react-description",
    verbose=True,
    handle_parsing_errors=True
)

# Step 12: Run Agent

In [ ]:
agent.run("Why do tickets increase during holidays?")